# Notebook 19 — Klasifikasi Arah 3 Kelas (Naik / Turun / Sideways)

Model LSTM klasifikasi arah harga **3 kelas** dibangun dari nol, dataset OHLCV + `sentiment_mean` (fitur *return*, sudah stasioner — reuse `load_return_features()` produksi, temuan Notebook 11). Alur: labeling 3 kelas dengan ambang *sideways* berbasis riwayat pengujian sebelumnya → Hyperparameter Tuning (Random Search) → training final dibandingkan pada split **80/20, 70/30, 60/40**.

**Catatan jujur:** lantai tebak-acak untuk 3 kelas adalah `ln(3) ≈ 1.0986` (BUKAN 0). Kalau `train_loss` tidak turun jauh di bawah itu, itu tanda sinyal lemah di data — bukan bug pipeline. Lihat Bagian 8 untuk cara membacanya.

In [9]:
from __future__ import annotations

import json
import logging
import sys
from copy import deepcopy
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("E:/forecasting_index_snp500/stock_prediction")
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("notebook19")

OUT_DIR = ROOT / "notebooks" / "outputs" / "notebook19"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 20
HORIZON = 5
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", device)

from pipeline.feature_engineering.dataset_builder import load_return_features, RETURN_FEATURE_COLS_B
from models.lstm.notebook07_common import _make_windows
from models.lstm.lstm_model import FlexibleLSTMClassifier

FEATURE_COLS = RETURN_FEATURE_COLS_B
print("FEATURE_COLS:", FEATURE_COLS)

16:30:45  INFO      Device: cuda


FEATURE_COLS: ['ret_open', 'ret_high', 'ret_low', 'ret_close', 'ret_volume', 'sentiment_mean']


## Bagian 1 — Load Data (OHLCV + `sentiment_mean`, sudah stasioner)

Reuse `load_return_features()` produksi — fitur *return* (`pct_change`), BUKAN harga level (temuan Notebook 11: harga level non-stasioner → 13.5% baris test di luar jangkauan scaling).

In [10]:
df_raw = load_return_features().sort_values("Date").reset_index(drop=True)
print(f"Total baris: {len(df_raw):,} ({df_raw['Date'].min().date()} s/d {df_raw['Date'].max().date()})")
df_raw.head()

16:30:45  INFO      Stock loaded: 1552 rows (2018-01-02 – 2024-03-04)
16:30:45  INFO      Sentiment loaded: 1528 rows (2018-01-02 – 2024-03-04)
16:30:45  INFO      Joined dataset: 1528 rows (2018-01-02 – 2024-03-04)
16:30:45  INFO      Return-features dataset: 1526 rows (2018-01-03 – 2024-03-04)


Total baris: 1,526 (2018-01-03 s/d 2024-03-04)


,Date,ret_open,ret_high,ret_low,ret_close,ret_volume,sentiment_mean,Close
0,2018-01-03,0.005261,0.006855,0.005745,0.006399,0.043150,-0.166667,2713.060059
1,2018-01-04,0.007954,0.005497,0.007895,0.004029,0.043259,0.000000,2723.989990
2,2018-01-05,0.004420,0.005188,0.003255,0.007034,-0.123889,0.000000,2743.149902
3,2018-01-08,0.004152,0.001844,0.003549,0.001662,0.002124,0.166667,2747.709961
4,2018-01-09,0.003092,0.003868,0.003748,0.001303,0.068173,0.666667,2751.290039


## Bagian 2 — Pelabelan 3 Kelas (Naik/Turun/Sideways) & Ambang Batas *Sideways*

Label dibentuk dari *forward return* `horizon` hari: **Naik** kalau return > θ, **Turun** kalau return < −θ, selain itu **Sideways**. Sebelum `θ` ditentukan, riwayat pengujian ambang *deadband* (horizon 1 hari) di notebook sebelumnya dibaca dulu:

| Ambang (θ) | Sumber | Hasil |
|---|---|---|
| 0.25% | Notebook 15 | AUC test 0.519, sisa data 957/1526 baris |
| 0.50% | Notebook 15 | AUC test 0.502, sisa data 868/1526 baris |
| 1.00% | Notebook 15 | AUC test 0.494, sisa data makin sedikit |
| 0.50% (grid θ×w_min) | Notebook 18 | **AUC test terbaik 0.648**, f1_macro 0.616 |
| 1.00–2.00% | Notebook 18 | AUC test turun ke 0.51–0.62, data makin sedikit |

**Kesimpulan riwayat:** θ = 0.5% adalah titik seimbang terbaik di eksperimen horizon 1 hari. Notebook ini pakai horizon **5 hari** (konsisten pipeline resmi saat ini), jadi ambang diskalakan dengan `sqrt(horizon)` (asumsi volatilitas random-walk, cara yang sama dipakai triple-barrier Notebook 16): `θ_5 = 0.5% × √5 ≈ 1.12%`. Distribusi 3 kelas hasil ambang ini dicek langsung di sel berikut — kalau timpang, `THETA_DAILY` tinggal diubah.

In [11]:
# Ambang batas sideways (hasil baca riwayat notebook 15/18 di atas)
THETA_DAILY = 0.005
THETA = THETA_DAILY * np.sqrt(HORIZON)
LABEL_NAMES = {0: "Turun", 1: "Sideways", 2: "Naik"}


# --- Proses labeling 3 kelas ---
def label_3class(df: pd.DataFrame, horizon: int, theta: float) -> pd.DataFrame:
    out = df.copy()
    fwd_return = out["Close"].shift(-horizon) / out["Close"] - 1.0
    out["fwd_return"] = fwd_return
    out["label"] = np.select([fwd_return > theta, fwd_return < -theta], [2, 0], default=1)
    out = out.iloc[:-horizon].copy().reset_index(drop=True)
    out["label"] = out["label"].astype("int64")
    return out


df_labeled = label_3class(df_raw, HORIZON, THETA)
counts = df_labeled["label"].value_counts().sort_index()
print(f"theta = {THETA:.4%}  (horizon={HORIZON} hari, {len(df_labeled)} baris)")
for k, v in counts.items():
    print(f"  {LABEL_NAMES[k]:>9s}: {v:4d}  ({v / len(df_labeled):.1%})")

theta = 1.1180%  (horizon=5 hari, 1521 baris)
      Turun:  348  (22.9%)
   Sideways:  613  (40.3%)
       Naik:  560  (36.8%)


## Bagian 3 — Split Kronologis (parametrized) + Windowing

Fungsi split & windowing dipakai ulang untuk Hyperparameter Tuning (70/15/15, butuh val) maupun training final (80/20, 70/30, 60/40, tanpa val). `MinMaxScaler` fit HANYA di train (anti-leakage).

In [12]:
def chronological_split(df: pd.DataFrame, train_ratio: float, val_ratio: float = 0.0):
    n = len(df)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    df_train = df.iloc[:n_train].reset_index(drop=True)
    df_val = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
    df_test = df.iloc[n_train + n_val:].reset_index(drop=True)
    return df_train, df_val, df_test


def build_arrays(df_train, df_val, df_test, feature_cols, window):
    """Dipakai HPO (butuh val)."""
    scaler = MinMaxScaler()
    train_feat = scaler.fit_transform(df_train[feature_cols].values.astype(np.float64))
    val_feat = scaler.transform(df_val[feature_cols].values.astype(np.float64))
    test_feat = scaler.transform(df_test[feature_cols].values.astype(np.float64))
    X_train, y_train = _make_windows(train_feat, df_train["label"].values, window)
    X_val, y_val = _make_windows(val_feat, df_val["label"].values, window)
    X_test, y_test = _make_windows(test_feat, df_test["label"].values, window)
    return {"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val,
            "X_test": X_test, "y_test": y_test, "scaler": scaler}


def build_arrays_train_test(df_train, df_test, feature_cols, window):
    """Dipakai training final (TANPA val — config sudah fixed, tak ada seleksi)."""
    scaler = MinMaxScaler()
    train_feat = scaler.fit_transform(df_train[feature_cols].values.astype(np.float64))
    test_feat = scaler.transform(df_test[feature_cols].values.astype(np.float64))
    X_train, y_train = _make_windows(train_feat, df_train["label"].values, window)
    X_test, y_test = _make_windows(test_feat, df_test["label"].values, window)
    return {"X_train": X_train, "y_train": y_train, "X_test": X_test, "y_test": y_test, "scaler": scaler}

## Bagian 4 — Fungsi Training (early stopping berbasis `val_f1_macro`)

Pelajaran dari Notebook 13 (akar penyebab *collapse*): early stopping HARUS memantau `val_f1_macro`, **bukan** `val_loss` — di tugas ber-SNR rendah, `val_loss` minimum di epoch 1-2 lalu naik, sehingga bobot yang tersimpan adalah model yang belum belajar. `min_epochs`/`patience` mencegah stop terlalu dini.

In [13]:
def build_optimizer(name: str, params, lr: float, weight_decay: float = 0.0):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Optimizer tidak dikenal: {name}")


def compute_class_weights_3(y_train: np.ndarray, device: torch.device, n_classes: int = 3) -> torch.Tensor:
    counts = np.bincount(y_train.astype(np.int64), minlength=n_classes)
    w = 1.0 / (counts + 1e-9)
    w = w / w.sum() * n_classes
    return torch.tensor(w, dtype=torch.float32, device=device)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n_samples = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        n_samples += xb.size(0)
    return total_loss / n_samples if n_samples else 0.0


def evaluate_multiclass(model, loader, criterion, device):
    model.eval()
    total_loss, n_samples = 0.0, 0
    y_true_all, y_pred_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            n_samples += xb.size(0)
            y_true_all.extend(yb.cpu().numpy().tolist())
            y_pred_all.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    y_true, y_pred = np.array(y_true_all), np.array(y_pred_all)
    avg_loss = total_loss / n_samples if n_samples else 0.0
    acc = accuracy_score(y_true, y_pred) if n_samples else 0.0
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0) if n_samples else 0.0
    return avg_loss, acc, f1_macro, y_true, y_pred


def train_with_early_stopping(model, train_loader, val_loader, optimizer, criterion, device,
                               max_epochs, min_epochs=20, patience=15):
    best_score, best_state, best_epoch, no_improve, stopped_epoch = -1.0, None, 0, 0, max_epochs
    history = {"train_loss": [], "val_loss": [], "val_f1_macro": []}
    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, _, val_f1_macro, _, _ = evaluate_multiclass(model, val_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1_macro"].append(val_f1_macro)
        if val_f1_macro > best_score + 1e-5:
            best_score, best_epoch, no_improve = val_f1_macro, epoch, 0
            best_state = deepcopy({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        elif epoch >= min_epochs:
            no_improve += 1
            if no_improve >= patience:
                stopped_epoch = epoch
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_epoch, best_score

## Bagian 5 — Hyperparameter Tuning (Random Search)

Dicari lewat split **70/15/15** (satu-satunya tahap yang butuh val terpisah untuk memilih config). Search space kecil & resep early stopping (`min_epochs=20`, `patience=15`) mengikuti perbaikan Notebook 13, supaya Random Search tidak terjebak pada config yang *collapse*.

In [14]:
def sample_config(rng: np.random.Generator) -> dict:
    log_lr = rng.uniform(np.log10(5e-4), np.log10(5e-3))
    return {
        "hidden_size": int(rng.choice([16, 32, 64])),
        "num_layers": int(rng.choice([1, 2])),
        "dropout": float(rng.uniform(0.0, 0.4)),
        "optimizer": str(rng.choice(["adam", "rmsprop"])),
        "activation": str(rng.choice(["relu", "tanh"])),
        "lr": float(10 ** log_lr),
        "batch_size": int(rng.choice([32, 64])),
        "weight_decay": float(rng.choice([0.0, 1e-5, 1e-4])),
    }


N_ITER = 15
MAX_EPOCHS = 150
MIN_EPOCHS = 20
PATIENCE = 15

df_tr_hpo, df_val_hpo, df_te_hpo = chronological_split(df_labeled, train_ratio=0.70, val_ratio=0.15)
ds_hpo = build_arrays(df_tr_hpo, df_val_hpo, df_te_hpo, FEATURE_COLS, WINDOW)
print(f"HPO split 70/15/15 -> train={len(ds_hpo['X_train'])} val={len(ds_hpo['X_val'])} test={len(ds_hpo['X_test'])}")

train_ds_hpo = TensorDataset(torch.from_numpy(ds_hpo["X_train"]), torch.from_numpy(ds_hpo["y_train"]))
val_ds_hpo = TensorDataset(torch.from_numpy(ds_hpo["X_val"]), torch.from_numpy(ds_hpo["y_val"]))

rng = np.random.default_rng(SEED)
best = {"val_f1_macro": -1.0, "config": None, "best_epoch": None}

for i in range(N_ITER):
    config = sample_config(rng)
    torch.manual_seed(SEED + i)
    model = FlexibleLSTMClassifier(
        input_size=len(FEATURE_COLS), hidden_size=config["hidden_size"], num_layers=config["num_layers"],
        dropout=config["dropout"], activation=config["activation"], num_classes=3,
    ).to(device)
    optimizer = build_optimizer(config["optimizer"], model.parameters(), config["lr"], config["weight_decay"])
    criterion = nn.CrossEntropyLoss(weight=compute_class_weights_3(ds_hpo["y_train"], device))

    train_loader = DataLoader(train_ds_hpo, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds_hpo, batch_size=config["batch_size"], shuffle=False)

    history, best_epoch, val_f1_macro = train_with_early_stopping(
        model, train_loader, val_loader, optimizer, criterion, device,
        max_epochs=MAX_EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE,
    )
    logger.info("iter %2d/%d  val_f1_macro=%.4f  min_train_loss=%.4f  best_epoch=%d",
                i + 1, N_ITER, val_f1_macro, min(history["train_loss"]), best_epoch)

    if val_f1_macro > best["val_f1_macro"]:
        best = {"val_f1_macro": val_f1_macro, "config": config, "best_epoch": best_epoch}

print("\n=== Best config (HPO) ===")
print(json.dumps(best, indent=2))

HPO split 70/15/15 -> train=1045 val=209 test=210


16:30:53  INFO      iter  1/15  val_f1_macro=0.3733  min_train_loss=1.0549  best_epoch=43
16:30:56  INFO      iter  2/15  val_f1_macro=0.1911  min_train_loss=1.0988  best_epoch=4
16:30:58  INFO      iter  3/15  val_f1_macro=0.3173  min_train_loss=1.0779  best_epoch=11
16:31:04  INFO      iter  4/15  val_f1_macro=0.3803  min_train_loss=1.0655  best_epoch=24
16:31:09  INFO      iter  5/15  val_f1_macro=0.3627  min_train_loss=1.0599  best_epoch=40
16:31:17  INFO      iter  6/15  val_f1_macro=0.3759  min_train_loss=1.0647  best_epoch=29
16:31:23  INFO      iter  7/15  val_f1_macro=0.3252  min_train_loss=1.0719  best_epoch=27
16:31:27  INFO      iter  8/15  val_f1_macro=0.3589  min_train_loss=1.0729  best_epoch=45
16:31:30  INFO      iter  9/15  val_f1_macro=0.3233  min_train_loss=1.0737  best_epoch=29
16:31:34  INFO      iter 10/15  val_f1_macro=0.3487  min_train_loss=1.0615  best_epoch=10
16:31:45  INFO      iter 11/15  val_f1_macro=0.3896  min_train_loss=1.0567  best_epoch=31
16:31:50  I


=== Best config (HPO) ===
{
  "val_f1_macro": 0.40416008991689284,
  "config": {
    "hidden_size": 32,
    "num_layers": 1,
    "dropout": 0.22609444259247555,
    "optimizer": "adam",
    "activation": "tanh",
    "lr": 0.002330091449155189,
    "batch_size": 64,
    "weight_decay": 1e-05
  },
  "best_epoch": 29
}


## Bagian 6 — Training Final: Perbandingan Split 80/20, 70/30, 60/40

Config hasil HPO dipakai **tetap** (tidak dicari ulang per rasio) supaya perbandingan murni karena rasio split. Epoch dikunci ke `best_epoch` dari HPO — training final tanpa val terpisah (config sudah fixed, tidak ada seleksi yang diambil dari metrik live, konsisten pola training manual di API produksi).

In [15]:
SPLIT_RATIOS = {"80/20": 0.80, "70/30": 0.70, "60/40": 0.60}
best_config = best["config"]
FINAL_EPOCHS = max(best["best_epoch"], 1)

results = {}
for name, train_ratio in SPLIT_RATIOS.items():
    df_tr, _, df_te = chronological_split(df_labeled, train_ratio=train_ratio, val_ratio=0.0)
    ds = build_arrays_train_test(df_tr, df_te, FEATURE_COLS, WINDOW)

    torch.manual_seed(SEED)
    model = FlexibleLSTMClassifier(
        input_size=len(FEATURE_COLS), hidden_size=best_config["hidden_size"], num_layers=best_config["num_layers"],
        dropout=best_config["dropout"], activation=best_config["activation"], num_classes=3,
    ).to(device)
    optimizer = build_optimizer(best_config["optimizer"], model.parameters(), best_config["lr"], best_config["weight_decay"])
    criterion = nn.CrossEntropyLoss(weight=compute_class_weights_3(ds["y_train"], device))

    train_loader = DataLoader(TensorDataset(torch.from_numpy(ds["X_train"]), torch.from_numpy(ds["y_train"])),
                               batch_size=best_config["batch_size"], shuffle=True)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(ds["X_test"]), torch.from_numpy(ds["y_test"])),
                              batch_size=best_config["batch_size"], shuffle=False)

    train_losses = [train_one_epoch(model, train_loader, optimizer, criterion, device) for _ in range(FINAL_EPOCHS)]

    test_loss, test_acc, test_f1_macro, y_true, y_pred = evaluate_multiclass(model, test_loader, criterion, device)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    recall_per_class = recall_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)

    results[name] = {
        "model": model, "scaler": ds["scaler"],
        "n_train": len(ds["X_train"]), "n_test": len(ds["X_test"]),
        "min_train_loss": float(min(train_losses)), "test_loss": test_loss, "test_accuracy": test_acc,
        "test_precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "test_recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "test_f1_macro": test_f1_macro,
        "recall_turun": float(recall_per_class[0]), "recall_sideways": float(recall_per_class[1]),
        "recall_naik": float(recall_per_class[2]), "confusion_matrix": cm.tolist(),
    }
    logger.info("[%s] train=%d test=%d min_train_loss=%.4f test_f1_macro=%.4f",
                name, results[name]["n_train"], results[name]["n_test"],
                results[name]["min_train_loss"], test_f1_macro)

16:32:03  INFO      [80/20] train=1197 test=286 min_train_loss=1.0643 test_f1_macro=0.3644
16:32:04  INFO      [70/30] train=1045 test=438 min_train_loss=1.0672 test_f1_macro=0.2401
16:32:06  INFO      [60/40] train=893 test=590 min_train_loss=1.0663 test_f1_macro=0.3809


## Bagian 7 — Tabel Perbandingan

In [16]:
comparison = pd.DataFrame({
    name: {k: v for k, v in r.items() if k not in ("model", "scaler", "confusion_matrix")}
    for name, r in results.items()
}).T
comparison_path = OUT_DIR / "notebook19_split_comparison.csv"
comparison.to_csv(comparison_path)
print(comparison.to_string())
print(f"\nDisimpan ke: {comparison_path}")

for name, r in results.items():
    print(f"\n=== Confusion matrix {name} (baris=aktual, kolom=prediksi; urutan Turun/Sideways/Naik) ===")
    print(pd.DataFrame(r["confusion_matrix"], index=["Turun", "Sideways", "Naik"], columns=["Turun", "Sideways", "Naik"]))

       n_train  n_test  min_train_loss  test_loss  test_accuracy  test_precision_macro  test_recall_macro  test_f1_macro  recall_turun  recall_sideways  recall_naik
80/20   1197.0   286.0        1.064335   1.072242       0.391608              0.403441           0.435454       0.364399      0.659574         0.500000     0.146789
70/30   1045.0   438.0        1.067229   1.087701       0.285388              0.438581           0.371886       0.240137      0.920000         0.130952     0.064706
60/40    893.0   590.0        1.066276   1.089981       0.383051              0.392925           0.393512       0.380870      0.496855         0.408451     0.275229

Disimpan ke: E:\forecasting_index_snp500\stock_prediction\notebooks\outputs\notebook19\notebook19_split_comparison.csv

=== Confusion matrix 80/20 (baris=aktual, kolom=prediksi; urutan Turun/Sideways/Naik) ===
          Turun  Sideways  Naik
Turun        31         9     7
Sideways     49        65    16
Naik         52        41    16



## Bagian 8 — Diagnosa: Apakah Model Benar-Benar Belajar?

Lantai tebak-acak 3 kelas = `ln(3) ≈ 1.0986`. Baca `min_train_loss` tiap rasio (tabel Bagian 7) relatif ke angka ini:

- **`train_loss` ≈ 1.0986 dan menetap** → model BELUM belajar apa-apa (bukan bug — kemungkinan sinyal terlalu lemah, konsisten temuan Notebook 09/11/13).
- **`train_loss` << 1.0986 tapi `test_f1_macro` rendah/`test_loss` tinggi** → overfitting, bukan underfitting — solusinya regularisasi (dropout/weight_decay), bukan menambah epoch.
- **`recall_turun`/`recall_sideways`/`recall_naik`** — kalau salah satunya ≈ 0 di ketiga rasio, model *collapse* ke kelas mayoritas (kemungkinan besar Sideways, kelas dengan sampel terbanyak).

In [17]:
LOSS_FLOOR = float(np.log(3))
for name, r in results.items():
    recalls = [r["recall_turun"], r["recall_sideways"], r["recall_naik"]]
    collapse = any(rc < 0.05 for rc in recalls)
    print(f"{name}: min_train_loss={r['min_train_loss']:.4f} (lantai={LOSS_FLOOR:.4f})  "
          f"recall(Turun/Sideways/Naik)={[round(x, 3) for x in recalls]}  collapse={collapse}")

80/20: min_train_loss=1.0643 (lantai=1.0986)  recall(Turun/Sideways/Naik)=[0.66, 0.5, 0.147]  collapse=False
70/30: min_train_loss=1.0672 (lantai=1.0986)  recall(Turun/Sideways/Naik)=[0.92, 0.131, 0.065]  collapse=False
60/40: min_train_loss=1.0663 (lantai=1.0986)  recall(Turun/Sideways/Naik)=[0.497, 0.408, 0.275]  collapse=False


## Bagian 9 — Simpan Artefak

In [18]:
for name, r in results.items():
    tag = name.replace("/", "_")
    torch.save({
        "model_state_dict": r["model"].state_dict(),
        "config": best_config,
        "feature_cols": FEATURE_COLS,
        "window": WINDOW,
        "horizon": HORIZON,
        "theta": THETA,
        "label_names": LABEL_NAMES,
    }, OUT_DIR / f"notebook19_split_{tag}.pt")
    joblib.dump(r["scaler"], OUT_DIR / f"notebook19_split_{tag}_scaler.joblib")

with open(OUT_DIR / "notebook19_best_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "config": best_config, "val_f1_macro": best["val_f1_macro"], "best_epoch": best["best_epoch"],
        "theta": THETA, "theta_daily": THETA_DAILY, "horizon": HORIZON, "window": WINDOW,
    }, f, indent=2)

print(f"Artefak tersimpan di: {OUT_DIR}")

Artefak tersimpan di: E:\forecasting_index_snp500\stock_prediction\notebooks\outputs\notebook19


## Penutup

Notebook ini murni membangun & membandingkan model — **belum dieksekusi**. HPO (15 iterasi × sampai 150 epoch) bisa makan waktu; pantau log per iterasi. Kalau hasil masih menunjukkan *collapse*/`train_loss` mentok di lantai `ln(3)`, opsi lanjutan: perbesar `N_ITER`, uji `THETA_DAILY` lain (mis. 0.25%/1%), atau tinjau ulang `HORIZON`.